# 01 — freeze which CholecT50 videos may train and which are the ruler

Rung 48 (infra step). Writes `experiments/splits/cholect50_split_v1.csv`: **35 train / 15 hold**
over CholecT50's 50 videos. **No GPU, no model, no challenge data.**

**Why this comes first.** The moment an arm trains on any CholecT50 frame, Strasbourg is inside
its distribution and the probe stops measuring **centre shift** — it measures **unseen scene**.
So the partition has to be frozen *before* a corpus is built, not after.

**The label rule this rests on**, adjudicated by eye on 2026-08-19 over 30 frames / 15 videos:

| | early frames | late frames |
|---|---:|---:|
| clip visible | **0** | **12** |
| no clip | 14 | 3 |
| unsure | 1 | 0 |

`clipper` alone is the *applier*, and the challenge says a clip counts *"only once placed"* —
read raw, the applier implies a clip only 41 % of the time. Read by TIME it is 80 %:
**`clipper` at or after the first `clipper,clip,*` triplet of the same video.** That rule is in
`_tools/cholect50.py` and it discards 122 of 3,429 clipper frames.

⚠️ The residual ~20 % is a constant every arm meets equally — **paired comparisons survive it,
absolute recall does not.**

In [ ]:
# --- bootstrap -------------------------------------------------------------------
import sys
from pathlib import Path

EXP = Path.cwd()
REPO = EXP
while REPO != REPO.parent and not ((REPO / ".git").exists() or (REPO / "src").is_dir()):
    REPO = REPO.parent
for p in (EXP / "_tools", REPO / "src"):
    if p.is_dir():
        sys.path.insert(0, str(p))

import pandas as pd
import cholect50 as T

In [ ]:
# --- config (inline, this cell IS the run) ---------------------------------------
CHOLECT50 = Path("/mnt/datos/code/ai/ORENA/surege-vlm/cholect50/extracted/CholecT50")
MANIFEST  = REPO / "experiments" / "splits" / "cholect50_split_v1.csv"
N_HOLD    = 15          # 30 % of the 50 — the ruler
SEED      = 42

pos = T.video_positives(CHOLECT50 / "labels")
print(f"{len(pos)} videos · {pos.n_frames.sum():,} frames · "
      f"{pos.n_pos.sum():,} positives (clip {pos.n_clip.sum():,} + bag {pos.n_bag.sum():,})")
print(f"per video: min {pos.n_pos.min()} · median {pos.n_pos.median():.0f} · max {pos.n_pos.max()}")

In [ ]:
# --- freeze it --------------------------------------------------------------------
split = T.build_split(pos, n_hold=N_HOLD, seed=SEED)
digest = T.write_manifest(split, MANIFEST)

g = split.groupby("split").agg(videos=("video_id", "count"), frames=("n_frames", "sum"),
                               clip=("n_clip", "sum"), bag=("n_bag", "sum"), pos=("n_pos", "sum"))
g["pct_pos"] = (100 * g.pos / g.pos.sum()).round(1)
print(g.to_string())
print(f"\nsha256 {digest[:12]}…  ->  {MANIFEST.name}")
print("\nHOLD (the ruler):", " ".join(sorted(split[split.split == 'hold'].video_id)))

In [ ]:
# --- guards: reload it the way every later notebook will ---------------------------
back = T.load_manifest(MANIFEST, verify=True)          # rejects a hand-edited manifest
assert len(back) == 50, len(back)
assert set(back.split) == {"train", "hold"}
assert (back.split == "hold").sum() == N_HOLD

share = back.groupby("split").n_pos.sum() / back.n_pos.sum()
for cls in ("n_clip", "n_bag"):
    s = back.groupby("split")[cls].sum() / back[cls].sum()
    print(f"{cls:7} hold share {s['hold']:.3f}")
print(f"{'n_pos':7} hold share {share['hold']:.3f}   (target {N_HOLD/50:.3f})")
print("\nall guards passed")